# 03 — Normalize to SLP1

1. Detect transliteration scheme per file (IAST or Devanagari expected).
2. Convert all text to SLP1 using `indic_transliteration`.
3. **Round-trip check**: SLP1 → original scheme → SLP1 on a random 1 000-line sample.
   If mismatch rate > 0.5%, STOP and show examples.
4. Output: `data/interim/corpus_slp1.jsonl` (all sources combined, pre-dedup).

In [1]:
import json, random, re, sys
from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm

from indic_transliteration import sanscript, detect as iast_detect
from indic_transliteration.sanscript import transliterate

BASE    = Path('/Users/sidharthbildikar/Desktop/code/llm-paninian-compression/sanskrit_corpus')
INTERIM = BASE / 'data' / 'interim'

GRETIL_JSONL = INTERIM / 'gretil_extracted.jsonl'
DCS_JSONL    = INTERIM / 'dcs_extracted.jsonl'
OUT_JSONL    = INTERIM / 'corpus_slp1.jsonl'

print('indic_transliteration version:', sanscript.__version__ if hasattr(sanscript, '__version__') else 'unknown')
print('Scheme constants:', sanscript.IAST, sanscript.DEVANAGARI, sanscript.SLP1)


indic_transliteration version: unknown
Scheme constants: iast devanagari slp1


## Scheme detection

Sample the first 500 chars of each unique file to detect its scheme.
We classify each line's source file as IAST or DEVANAGARI.
Files that can't be classified are flagged and excluded.

In [2]:
_DEVA_RE  = re.compile(r'[ऀ-ॿ]')
_LATIN_RE = re.compile(r'[A-Za-z]')

# Precomposed IAST diacritics specific to Sanskrit romanization
_IAST_CHARS = set('āīūṛṝḷṃḥśṣṭḍṅñṇḻĀĪŪṚṜḶṂḤŚṢṬḌṄÑṆḺ')

# Uniquely SLP1 characters: f=ṛ, w=ṭ, q=ḍ — absent from both IAST and HK
_SLP1_UNIQUE_RE = re.compile(r'[fwq]')
# Uppercase letter immediately after a lowercase letter = SLP1/HK phoneme mid-word.
# IAST uses Unicode diacritics (ā ī ū ṃ ḥ …) rather than ASCII uppercase for these.
_MID_UPPER_RE = re.compile(r'[a-z][A-Z]')


def detect_scheme(text: str) -> str:
    """Return 'DEVANAGARI', 'IAST', 'SLP1', or 'UNKNOWN'."""
    if _DEVA_RE.search(text):
        return 'DEVANAGARI'
    has_iast = any(c in _IAST_CHARS for c in text)
    if has_iast:
        return 'IAST'
    if not _LATIN_RE.search(text):
        return 'UNKNOWN'
    # Pure ASCII Latin: check for SLP1/HK signals.
    # SLP1 unique chars (f/w/q) are definitive; mid-word uppercase (M for anusvara,
    # A for ā, B for bh, etc.) is present in both SLP1 and HK but never in true IAST.
    if _SLP1_UNIQUE_RE.search(text) or _MID_UPPER_RE.search(text):
        return 'SLP1'
    # Pure lowercase ASCII — rare in practice; treat as simplified IAST.
    return 'IAST'


_test_cases = [
    ('agnim īḻe purohitam',                          'IAST'),
    ('अग्निमीळे पुरोहितम्',                          'DEVANAGARI'),
    ('na BadrakamidaM nATA na kartavyaM punarmayA',   'SLP1'),
    ('agniM ile purohitam',                           'SLP1'),   # M after lowercase = mid-upper
    ('agne naya supathA rAye',                        'SLP1'),   # A after lowercase
    ('agnim ile purohitam',                           'IAST'),   # pure lowercase → IAST fallback
]
print('detect_scheme tests:')
for txt, expected in _test_cases:
    got = detect_scheme(txt)
    print(f'  {txt[:42]!r:44s} -> {got} [{"OK" if got==expected else "FAIL expected "+expected}]')


detect_scheme tests:
  'agnim īḻe purohitam'                        -> IAST [OK]
  'अग्निमीळे पुरोहितम्'                        -> DEVANAGARI [OK]
  'na BadrakamidaM nATA na kartavyaM punarmay' -> SLP1 [OK]
  'agniM ile purohitam'                        -> SLP1 [OK]
  'agne naya supathA rAye'                     -> SLP1 [OK]
  'agnim ile purohitam'                        -> IAST [OK]


## Detect scheme per source file (using representative sample of lines)

In [3]:
def build_file_scheme_map(jsonl_path: Path, sample_lines: int = 20) -> dict:
    """Return {filename: scheme_str} for each unique file in the JSONL."""
    file_samples: dict = {}
    with jsonl_path.open(encoding='utf-8') as f:
        for line in f:
            rec = json.loads(line)
            fn  = rec['file']
            if fn not in file_samples:
                file_samples[fn] = []
            if len(file_samples[fn]) < sample_lines:
                file_samples[fn].append(rec['text'])

    scheme_map = {}
    unknown    = []
    for fn, texts in file_samples.items():
        combined = ' '.join(texts)
        s = detect_scheme(combined)
        scheme_map[fn] = s
        if s == 'UNKNOWN':
            unknown.append(fn)

    scheme_counts = Counter(scheme_map.values())
    return scheme_map, scheme_counts, unknown

print('Building scheme map for GRETIL...')
g_scheme_map, g_counts, g_unknown = build_file_scheme_map(GRETIL_JSONL)
print(f'  GRETIL scheme distribution: {dict(g_counts)}')
if g_unknown:
    print(f'  UNKNOWN files ({len(g_unknown)}): {g_unknown[:5]}')

print('Building scheme map for DCS...')
d_scheme_map, d_counts, d_unknown = build_file_scheme_map(DCS_JSONL)
print(f'  DCS scheme distribution: {dict(d_counts)}')
if d_unknown:
    print(f'  UNKNOWN files ({len(d_unknown)}): {d_unknown[:5]}')

Building scheme map for GRETIL...


  GRETIL scheme distribution: {'IAST': 781}
Building scheme map for DCS...


  DCS scheme distribution: {'IAST': 15790}


## Convert all lines to SLP1

In [4]:
_SCHEME_CONST = {
    'IAST':       sanscript.IAST,
    'DEVANAGARI': sanscript.DEVANAGARI,
    'HK':         sanscript.HK,
    # SLP1 is handled specially (pass-through); not in this dict.
}


def convert_to_slp1(text: str, src_scheme: str) -> str:
    if src_scheme == 'SLP1':
        return text  # already SLP1, no conversion needed
    sc = _SCHEME_CONST.get(src_scheme)
    if sc is None:
        return None  # UNKNOWN — skip
    return transliterate(text, sc, sanscript.SLP1)


def process_jsonl(in_path: Path, scheme_map: dict) -> tuple:
    """Convert all records to SLP1. Returns (list_of_dicts, n_skipped)."""
    records, skipped = [], 0
    with in_path.open(encoding='utf-8') as f:
        for raw_line in f:
            rec    = json.loads(raw_line)
            scheme = scheme_map.get(rec['file'], 'UNKNOWN')
            slp1   = convert_to_slp1(rec['text'], scheme)
            if slp1 is None or not slp1.strip():
                skipped += 1
                continue
            rec['text_slp1']     = slp1
            rec['src_scheme']    = scheme
            rec['text_original'] = rec.pop('text')
            records.append(rec)
    return records, skipped


print('Converting GRETIL...')
g_records, g_skip = process_jsonl(GRETIL_JSONL, g_scheme_map)
print(f'  {len(g_records):,} lines converted, {g_skip} skipped (UNKNOWN or empty)')

print('Converting DCS...')
d_records, d_skip = process_jsonl(DCS_JSONL, d_scheme_map)
print(f'  {len(d_records):,} lines converted, {d_skip} skipped')


Converting GRETIL...


  1,519,182 lines converted, 0 skipped (UNKNOWN or empty)
Converting DCS...


  754,726 lines converted, 0 skipped


## Round-trip check (NON-NEGOTIABLE)

SLP1 → original_scheme → SLP1.  Must equal original SLP1.
Sample 1 000 lines randomly across both sources.
**Stops execution if mismatch rate > 0.5%.**

In [5]:
random.seed(42)
all_records = g_records + d_records
sample_size = min(1000, len(all_records))
sample      = random.sample(all_records, sample_size)

mismatches   = []
skipped_slp1 = 0
for rec in sample:
    src_scheme = rec['src_scheme']
    if src_scheme == 'SLP1':
        skipped_slp1 += 1
        continue  # already SLP1; no round-trip needed
    slp1_orig = rec['text_slp1']
    sc_const  = _SCHEME_CONST.get(src_scheme)
    if sc_const is None:
        continue
    back_orig = transliterate(slp1_orig, sanscript.SLP1, sc_const)
    back_slp1 = transliterate(back_orig, sc_const, sanscript.SLP1)
    if back_slp1 != slp1_orig:
        mismatches.append({
            'file':      rec['file'],
            'original':  rec['text_original'],
            'slp1':      slp1_orig,
            'roundtrip': back_slp1,
        })

checked       = sample_size - skipped_slp1
mismatch_rate = len(mismatches) / checked if checked else 0.0
print(f'Round-trip check on {sample_size} lines ({skipped_slp1} SLP1 pass-throughs skipped):')
print(f'  Checked    : {checked}')
print(f'  Mismatches : {len(mismatches)}')
print(f'  Mismatch % : {100*mismatch_rate:.3f}%')

if mismatches:
    print('\nFailing examples (first 10):')
    for ex in mismatches[:10]:
        print(f"  file     : {ex['file']}")
        print(f"  original : {ex['original'][:80]}")
        print(f"  slp1     : {ex['slp1'][:80]}")
        print(f"  roundtrip: {ex['roundtrip'][:80]}")
        print()

THRESHOLD = 0.005  # 0.5%
if mismatch_rate > THRESHOLD:
    raise RuntimeError(
        f'STOP: Round-trip mismatch rate {100*mismatch_rate:.3f}% exceeds 0.5% threshold. '
        f'Transliteration is LOSSY — inspect failing examples above.'
    )
else:
    print(f'Round-trip check PASSED (rate {100*mismatch_rate:.3f}% <= 0.5%)')


Round-trip check on 1000 lines (0 SLP1 pass-throughs skipped):
  Checked    : 1000
  Mismatches : 0
  Mismatch % : 0.000%
Round-trip check PASSED (rate 0.000% <= 0.5%)


## Write combined SLP1 JSONL (pre-dedup)

In [ ]:
# ── post-conversion cleaning ──────────────────────────────────────────────
# TARGET: pure SLP1 text.
# Valid characters: A-Z a-z + space + ' (avagraha) + . (daṇḍa) + | (daṇḍa alt)
_NON_ASCII_RE  = re.compile(r'[^\x00-\x7F]')
_START_END_RE  = re.compile(r'^\s*(?:start|end)\s+[A-Za-z]', re.IGNORECASE)
_PERCENT_RE    = re.compile(r'%')          # metrical annotation → drop line
_DOLLAR_RE     = re.compile(r'\$')         # astronomical encoding → drop line
_NESTED_BRACK  = re.compile(r'\w\[')       # word[grammar] annotation → drop line
_SUTRA_REF_RE  = re.compile(r'^\s*(?:ap|kaj)\d')  # Apastamba/Kautilya sutra lines → drop

_AT_WORD_RE    = re.compile(r'@\S*')
_BRACKET_REF   = re.compile(r'\[[^\]]{0,80}\]|\[[\d./-]+')
_CURLY_ANY     = re.compile(r'\{[^}]{0,80}\}')
_PAGE_REF      = re.compile(r':[a-zA-Z]\s*\d+')
_CARET         = re.compile(r'\^')
_LEAD_VREF     = re.compile(r'^[a-z]{1,3}\.?\d[\d.]*[a-z]{0,3}\.?\s+')
_LEAD_NUM      = re.compile(r'^\d[\d.\s]*\s(?=[A-Za-z])')
_TRAIL_NUMS    = re.compile(r'(\s+\d+)+\s*$')
_TRAIL_HYPHEN  = re.compile(r'\s*-\s*$')

# Strip leading grammar/prosody abbreviation runs: "p. ", "p. r. y. ", etc.
# Single lowercase letter + period + space is always editorial shorthand in these
# texts, never the start of Sanskrit running prose.
_ABBREV_RUN    = re.compile(r'^(?:[a-z]\.\s+)+')

# WHITELIST — only characters allowed in final SLP1 segment
_WHITELIST_RE  = re.compile(r"[^A-Za-z .'\|]")   # strips ?,(),*,0-9,-,,,;,",: etc.
_MULTI_PUNCT   = re.compile(r'[.\|]{2,}')          # collapse dot/pipe runs

_PREF   = r'[A-Za-z][A-Za-z0-9]*(?:_[A-Za-z0-9,.;/*\-]+)+'
_PSEP   = r'(?:\|{1,3}|\.{2,}|/{1,2}|_+)'
_P_ONLY = re.compile(r'^\s*(?:' + _PSEP + r'\s*)?\(?' + _PREF + r'\)?\s*(?:' + _PSEP + r'|:)?\s*$')
_P_ANY  = re.compile(r'\s*' + _PSEP + r'\s*\(?' + _PREF + r'\)?\s*(?:(?:\|{1,3}|\.{2,}|/{1,2}))?\s*')
_P_TSEP = re.compile(r'\s*(?:\|+|\.{2,}|/{1,2})\s*$')


def clean_slp1(text: str) -> list:
    """Return 0-N clean SLP1 segments. Returns [] for noise/markup lines."""
    if _NON_ASCII_RE.search(text):  return []
    if _START_END_RE.match(text):   return []
    if _PERCENT_RE.search(text):    return []
    if _DOLLAR_RE.search(text):     return []
    if _NESTED_BRACK.search(text):  return []
    if _SUTRA_REF_RE.match(text):   return []

    text = _AT_WORD_RE.sub('', text)
    text = _CARET.sub('', text)

    result = []
    for part in re.split(r'\s*&\s*', text):
        part = _BRACKET_REF.sub(' ', part)
        part = _CURLY_ANY.sub(' ', part)
        part = _PAGE_REF.sub('', part)
        part = _LEAD_VREF.sub('', part)
        part = _LEAD_NUM.sub('', part)
        # Three-pass trailing cleanup handles patterns like "55 - 70" or "0 -"
        part = _TRAIL_NUMS.sub('', part)
        part = _TRAIL_HYPHEN.sub('', part)
        part = _TRAIL_NUMS.sub('', part)
        part = re.sub(r'\s{2,}', ' ', part).strip()
        if not part: continue
        if _P_ONLY.match(part): continue
        for seg in _P_ANY.split(part):
            seg = _P_TSEP.sub('', seg)
            seg = _WHITELIST_RE.sub('', seg)      # enforce SLP1 character whitelist
            seg = _MULTI_PUNCT.sub(' ', seg)       # collapse .. / || runs to space
            seg = re.sub(r'\s{2,}', ' ', seg).strip()
            seg = seg.strip(".'|")                 # strip leading/trailing punct
            seg = _ABBREV_RUN.sub('', seg).strip() # strip leading abbreviation runs
            if seg and len(seg) >= 10 and re.search(r'[A-Za-z]', seg):
                result.append(seg)
    return result


# ── smoke tests for abbreviation stripping ────────────────────────────────
_abbrev_tests = [
    # Abbreviation prefix stripped, Sanskrit preserved
    ('p. nIcEranudAttaH',             ['nIcEranudAttaH']),
    ('p. samAhAraH svaratiH',         ['samAhAraH svaratiH']),
    # Multi-abbreviation run stripped
    ('p. r. y. Bilakzyate . udIcya',  ['Bilakzyate']),  # 'udIcya' < 10 chars → dropped
    # Too short after strip → dropped by length filter
    ('p. pratyayaH',                  []),
    ('p. paraSca',                    []),
    # Normal Sanskrit untouched
    ('aTa SabdAnuSAsanam .',          ['aTa SabdAnuSAsanam']),
]
print('Abbreviation-stripping smoke tests:')
all_ok = True
for inp, expected in _abbrev_tests:
    got = clean_slp1(inp)
    ok  = got == expected
    all_ok = all_ok and ok
    print(f'  {"OK" if ok else "FAIL"} | {inp!r}')
    if not ok:
        print(f'        got:      {got}')
        print(f'        expected: {expected}')
print('All OK\n' if all_ok else 'SOME FAILED\n')


n_in = n_dropped = n_split = n_written = 0
with OUT_JSONL.open('w', encoding='utf-8') as f:
    for rec in all_records:
        n_in += 1
        segs = clean_slp1(rec['text_slp1'])
        if not segs:
            n_dropped += 1
            continue
        if len(segs) > 1:
            n_split += 1
        for seg in segs:
            n_written += 1
            f.write(json.dumps({
                'text_id'   : rec['text_id'],
                'source'    : rec['source'],
                'file'      : rec['file'],
                'src_scheme': rec['src_scheme'],
                'text'      : seg,
            }, ensure_ascii=False) + '\n')

print(f'Post-conversion cleaning:')
print(f'  Input records               : {n_in:,}')
print(f'  Dropped (noise/markup)      : {n_dropped:,}')
print(f'  Lines split into segments   : {n_split:,}')
print(f'  Written to corpus_slp1.jsonl: {n_written:,}')


## SLP1 character inventory

In [7]:
from collections import Counter

char_counter = Counter()
for rec in all_records:
    char_counter.update(rec['text_slp1'])

# Separate Sanskrit phoneme chars from punctuation/ASCII
# SLP1 phonemes are defined here: https://www.sanskrit-lexicon.uni-koeln.de/talkMay2008/mak.pdf
SLP1_PHONEMES = set('aAiIuUeEoOfFxXqQRkKgGNcCjJYwWqQwWtTdDnpPbBmyrlvSzshHML')
inv_phonemes = sorted(c for c in char_counter if c in SLP1_PHONEMES)
inv_other    = sorted(c for c in char_counter if c not in SLP1_PHONEMES)

print(f'SLP1 phoneme characters in corpus ({len(inv_phonemes)}): {" ".join(inv_phonemes)}')
print(f'Non-phoneme characters ({len(inv_other)}): {" ".join(repr(c) for c in inv_other[:30])}')
if len(inv_other) > 30:
    print(f'  ... and {len(inv_other)-30} more')

SLP1 phoneme characters in corpus (50): A B C D E F G H I J K L M N O P Q R S T U W X Y a b c d e f g h i j k l m n o p q r s t u v w x y z
Non-phoneme characters (104): ' ' '!' '"' '#' '$' '%' '&' "'" '(' ')' '*' '+' ',' '-' '.' '0' '1' '2' '3' '4' '5' '6' '7' '8' '9' ':' ';' '<' '=' '>'
  ... and 74 more
